In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Evaluation
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Clustering Models
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn_extra.cluster import KMedoids

import warnings
warnings.filterwarnings('ignore')
import kagglehub
path = kagglehub.dataset_download("muhammadshefingani/and-x1f1ee-and-x1f1e9-indonesia-investment-2010-2025")
path2 = kagglehub.dataset_download("dannytheodore/socio-economic-of-indonesia-in-2021")
print("Path to dataset files:", path,path2)

Path to dataset files: C:\Users\mwils\.cache\kagglehub\datasets\muhammadshefingani\and-x1f1ee-and-x1f1e9-indonesia-investment-2010-2025\versions\1 C:\Users\mwils\.cache\kagglehub\datasets\dannytheodore\socio-economic-of-indonesia-in-2021\versions\2


In [13]:
import os
import pandas as pd

file_investasi = os.path.join(path, "investasi_2010-2025.csv")
file_sosial = os.path.join(path2, "2021socio_economic_indonesia.csv")

df_bkpm = pd.read_csv(file_investasi)
df_bps = pd.read_csv(file_sosial)


In [14]:
df_bkpm.head()

,period,investment_status,region,country,main_sector,sector_name,kbli_2digit,province,district_city,java_outside_java,island,investment_idr_million,investment_usd_thousand,indonesian_workers,year,quarter
0,2010-Q1,PMA,Afrika,Mauritania,Sektor Sekunder,Industri Makanan,(10-2015) Industri makanan,Kalimantan Tengah,Kabupaten Seruyan,Luar Jawa,Kalimantan,279631.2,29748.0,0,2010,Q1
1,2010-Q1,PMA,Afrika,Mauritius,Sektor Tersier,Perdagangan dan Reparasi,"(46-2015) Perdagangan besar, bukan mobil dan s...",Nusa Tenggara Barat,Kota Mataram,Luar Jawa,Bali dan Nusa Tenggara,0.0,0.0,44,2010,Q1
2,2010-Q1,PMA,Afrika,Seychelles,Sektor Primer,Pertambangan,(06-2015) Pertambangan minyak bumi dan gas ala...,Riau,Kabupaten Bengkalis,Luar Jawa,Sumatera,2265.3,241.0,692,2010,Q1
3,2010-Q1,PMA,Afrika,Seychelles,Sektor Primer,"Tanaman Pangan, Perkebunan, dan Peternakan","(01-2015) Pertanian tanaman, peternakan, perbu...",Kalimantan Barat,Kabupaten Sanggau,Luar Jawa,Kalimantan,0.0,0.0,110,2010,Q1
4,2010-Q1,PMA,Afrika,Seychelles,Sektor Primer,"Tanaman Pangan, Perkebunan, dan Peternakan","(01-2015) Pertanian tanaman, peternakan, perbu...",Nusa Tenggara Barat,Kabupaten Lombok Barat,Luar Jawa,Bali dan Nusa Tenggara,0.0,0.0,90,2010,Q1


In [15]:
df_bps.head()

,province,cities_reg,poorpeople_percentage,reg_gdp,life_exp,avg_schooltime,exp_percap
0,Aceh,Simeulue,18.98,2.275,65.240,9.48,7148
1,Aceh,Aceh Singkil,20.36,2.425,67.355,8.68,8776
2,Aceh,Aceh Selatan,13.18,5.531,64.360,8.88,8180
3,Aceh,Aceh Tenggara,13.41,5.063,68.155,9.67,8030
4,Aceh,Aceh Timur,14.45,10.616,68.705,8.21,8577


In [16]:
df_bkpm.isnull().sum()

period                     0
investment_status          0
region                     0
country                    0
main_sector                0
sector_name                0
kbli_2digit                0
province                   0
district_city              0
java_outside_java          0
island                     0
investment_idr_million     0
investment_usd_thousand    0
indonesian_workers         0
year                       0
quarter                    0
dtype: int64

In [17]:
df_bps.isnull().sum()

province                 0
cities_reg               0
poorpeople_percentage    0
reg_gdp                  0
life_exp                 0
avg_schooltime           0
exp_percap               0
dtype: int64

In [18]:
df_merged = pd.merge(df_bps, df_bkpm, on='province', how='inner')

In [19]:
# Menyamakan format nama provinsi sebelum merge
df_bkpm['province'] = df_bkpm['province'].str.strip().str.upper()
df_bps['province'] = df_bps['province'].str.strip().str.upper()

# Merangkum indikator sosial-ekonomi per provinsi
df_sosial_provinsi = (
    df_bps.groupby('province')
    .agg({
        'poorpeople_percentage': 'mean',
        'reg_gdp': 'mean',
        'life_exp': 'mean',
        'avg_schooltime': 'mean',
        'exp_percap': 'mean',
        'cities_reg': 'nunique'
    })
    .rename(columns={'cities_reg': 'jumlah_kabupaten_kota'})
)

# Merangkum investasi dan jumlah tenaga kerja per provinsi/status investasi
df_investasi_provinsi = (
    df_bkpm.groupby(['province', 'investment_status'])
    .agg({
        'investment_idr_million': 'sum',
        'indonesian_workers': 'sum'
    })
    .unstack(fill_value=0)
)
df_investasi_provinsi.columns = [
    f'{metric}_{status}'
    for metric, status in df_investasi_provinsi.columns
]

# Menggabungkan data sosial-ekonomi dan investasi asli
df_merged = df_sosial_provinsi.join(df_investasi_provinsi, how='inner')
df_merged.head()

,poorpeople_percentage,reg_gdp,life_exp,avg_schooltime,exp_percap,jumlah_kabupaten_kota,investment_idr_million_PMA,investment_idr_million_PMDN,indonesian_workers_PMA,indonesian_workers_PMDN
province,,,,,,,,,,
ACEH,15.693913,7.241043,68.256739,9.503478,9642.782609,23,1.903593e+07,6.223486e+07,29805,165261
BALI,4.915556,25.056778,72.673333,8.642222,13704.666667,9,1.316104e+08,6.468336e+07,236586,135673
BANTEN,6.643750,78.916125,68.155625,8.907500,12124.125000,8,5.543689e+08,3.142458e+08,1009155,645077
BENGKULU,14.929000,7.338900,67.604500,8.612000,10222.100000,10,1.461413e+07,4.638733e+07,44171,59646
GORONTALO,15.738333,6.981667,67.989167,7.900000,10068.833333,6,9.402148e+06,2.020974e+07,20494,33263


In [20]:
# Menangani missing values jika ada (Imputasi menggunakan median)
df_merged.fillna(df_merged.median(), inplace=True)

# Transformasi Data: Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_merged)
X_scaled_df = pd.DataFrame(X_scaled, columns=df_merged.columns, index=df_merged.index)
X_scaled_df.head()

,poorpeople_percentage,reg_gdp,life_exp,avg_schooltime,exp_percap,jumlah_kabupaten_kota,investment_idr_million_PMA,investment_idr_million_PMDN,indonesian_workers_PMA,indonesian_workers_PMDN
province,,,,,,,,,,
ACEH,0.720707,-0.649582,-0.411635,1.480579,-0.404935,0.836513,-0.677030,-0.448935,-0.553639,-0.357234
BALI,-1.132886,-0.091136,1.424051,0.241006,2.036663,-0.771890,-0.205403,-0.435041,-0.185337,-0.436123
BANTEN,-0.835682,1.597119,-0.453662,0.622810,1.086599,-0.886775,1.565727,0.981108,1.190701,0.922077
BENGKULU,0.589162,-0.646515,-0.682728,0.197508,-0.056708,-0.657004,-0.695554,-0.538863,-0.528052,-0.638830
GORONTALO,0.728346,-0.657712,-0.522847,-0.827246,-0.148836,-1.116547,-0.717390,-0.687408,-0.570223,-0.709174


In [21]:
n_clusters = 3 # Asumsi 3 tier: Maju, Berkembang, Tertinggal

# 1. K-Means
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
df_merged['Cluster_KMeans'] = kmeans.fit_predict(X_scaled)

# 2. Hierarchical Clustering (Agglomerative)
hc = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
df_merged['Cluster_HC'] = hc.fit_predict(X_scaled)

# 3. DBSCAN
# Parameter eps dan min_samples perlu di-tuning (hyperparameter tuning) bergantung pada sebaran data asli
dbscan = DBSCAN(eps=1.5, min_samples=2)
df_merged['Cluster_DBSCAN'] = dbscan.fit_predict(X_scaled)

# 4. Gaussian Mixture Models (GMM)
gmm = GaussianMixture(n_components=n_clusters, random_state=42)
df_merged['Cluster_GMM'] = gmm.fit_predict(X_scaled)

# 5. K-Medoids (PAM)
kmedoids = KMedoids(n_clusters=n_clusters, random_state=42)
df_merged['Cluster_KMedoids'] = kmedoids.fit_predict(X_scaled)

In [23]:
model_columns = {
    'KMeans': 'Cluster_KMeans',
    'HC': 'Cluster_HC',
    'DBSCAN': 'Cluster_DBSCAN',
    'GMM': 'Cluster_GMM',
    'KMedoids': 'Cluster_KMedoids'
}

evaluation_results = []

for model, col_name in model_columns.items():
    labels = df_merged[col_name]

    # DBSCAN mungkin menghasilkan 1 cluster (-1 untuk noise), perlu validasi agar metric bisa dihitung
    if len(set(labels)) > 1:
        sil_score = silhouette_score(X_scaled, labels)
        db_score = davies_bouldin_score(X_scaled, labels)
    else:
        sil_score = np.nan
        db_score = np.nan

    evaluation_results.append({
        'Model': model,
        'Silhouette Score (Higher is Better)': sil_score,
        'Davies-Bouldin Index (Lower is Better)': db_score
    })

eval_df = pd.DataFrame(evaluation_results)
eval_df.sort_values(by='Silhouette Score (Higher is Better)', ascending=False)

KeyError: 'Cluster_kmedoids'

In [ ]:
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler
import requests
import folium

# Skor kebutuhan: semakin tinggi, semakin membutuhkan prioritas pembangunan.
score_features = {
    'poorpeople_percentage': 1,
    'reg_gdp': -1,
    'life_exp': -1,
    'avg_schooltime': -1,
    'exp_percap': -1,
    'investment_idr_million_PMA': -1,
    'investment_idr_million_PMDN': -1
}

score_data = df_merged[list(score_features)].copy()
normalized = pd.DataFrame(
    MinMaxScaler().fit_transform(score_data),
    columns=score_data.columns,
    index=score_data.index
)

for column, direction in score_features.items():
    if direction == -1:
        normalized[column] = 1 - normalized[column]

priority_df = pd.DataFrame(index=df_merged.index)
priority_df['Skor_Kebutuhan'] = normalized.mean(axis=1) * 100
priority_df['Prioritas'] = pd.cut(
    priority_df['Skor_Kebutuhan'],
    bins=[-1, 33.33, 66.67, 101],
    labels=['Rendah', 'Sedang', 'Tinggi']
)
priority_df = priority_df.sort_values('Skor_Kebutuhan', ascending=False)

display(priority_df.head(10).round(2))

# Peta interaktif provinsi Indonesia.
geojson_url = 'https://raw.githubusercontent.com/superpikar/indonesia-geojson/master/indonesia.geojson'
response = requests.get(geojson_url, timeout=30)
response.raise_for_status()
geojson = response.json()

province_names = set(df_merged.index.str.upper())
property_keys = geojson['features'][0]['properties'].keys()
province_key = max(
    property_keys,
    key=lambda key: len({
        str(feature['properties'].get(key, '')).strip().upper()
        for feature in geojson['features']
    } & province_names)
)

for feature in geojson['features']:
    feature['properties'][province_key] = str(
        feature['properties'][province_key]
    ).strip().upper()

map_data = priority_df.reset_index()
map_data['province'] = map_data['province'].str.upper()

m = folium.Map(location=[-2.5, 118], zoom_start=5, tiles='CartoDB positron')
folium.Choropleth(
    geo_data=geojson,
    data=map_data,
    columns=['province', 'Skor_Kebutuhan'],
    key_on=f'feature.properties.{province_key}',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.25,
    legend_name='Skor Kebutuhan (semakin tinggi semakin prioritas)'
).add_to(m)

folium.LayerControl().add_to(m)
display(m)

ModuleNotFoundError: No module named 'folium'